# Module 15 — Inverse S-box and Reversibility

**Mathematics of Cryptography · Volume 1**

---

## Overview

AES is a block cipher — it must **encrypt and decrypt**. The S-box, which gives AES its non-linearity, must therefore be reversible. This notebook constructs the **inverse S-box** from the ground up, proves that it correctly reverses the forward S-box for every possible byte, and illuminates the structure that makes reversibility possible.

The forward S-box applies two stages:
1. **Stage 1** — GF(2⁸) multiplicative inversion
2. **Stage 2** — Affine transformation (linear bit-mixing + XOR with constant `0x63`)

Because both stages are individually invertible, the inverse S-box undoes them in **reverse order**:
1. **Stage 1** — Inverse affine transformation
2. **Stage 2** — GF(2⁸) multiplicative inversion again (since `(a⁻¹)⁻¹ = a`)

### Sections

1. Setup — GF(2⁸) helpers and forward S-box (carried from Module 14)
2. Building the inverse S-box table by table inversion
3. The inverse affine transformation — derivation and implementation
4. Step-by-step trace of the inverse S-box procedure
5. Proof: `InvSBOX[SBOX[a]] == a` for all 256 bytes
6. Bijection verification — INVSBOX is also a permutation
7. The shared intermediate value — forward and reverse paths compared
8. Interactive explorer: `round_trip_explore(hex_val)`
9. Collision thought experiment — what breaks when outputs collide
10. Summary and bridge to Module 16

In [ ]:
# ============================================================
# Section 1 — Setup: GF(2^8) helpers and forward S-box
# ============================================================
# These are carried directly from Module 14.
# All arithmetic is modulo m(x) = x^8 + x^4 + x^3 + x + 1 (0x11B).

def xtime(a: int) -> int:
    """Multiply a by x in GF(2^8) with reduction."""
    result = (a << 1) & 0xFF
    return result ^ 0x1B if (a & 0x80) else result


def gf_mul(a: int, b: int) -> int:
    """Multiply a and b in GF(2^8) using the peasant method."""
    result, aa = 0, a
    for i in range(8):
        if (b >> i) & 1:
            result ^= aa
        aa = xtime(aa)
    return result


def gf_inv(a: int) -> int:
    """Multiplicative inverse in GF(2^8). By convention gf_inv(0) = 0."""
    if a == 0:
        return 0
    for b in range(1, 256):
        if gf_mul(a, b) == 1:
            return b
    raise ValueError(f"No inverse for {a:#04x}")


# Affine constant: bits of 0x63, index 0 = LSB
CONST_FWD = [1, 1, 0, 0, 0, 1, 1, 0]


def affine_fwd(inv_byte: int) -> int:
    """Apply the AES forward affine transformation."""
    b = [(inv_byte >> i) & 1 for i in range(8)]
    s = [b[i] ^ b[(i+4)%8] ^ b[(i+5)%8] ^ b[(i+6)%8] ^ b[(i+7)%8] ^ CONST_FWD[i]
         for i in range(8)]
    return sum(s[i] << i for i in range(8))


def sbox(a: int) -> int:
    """Forward AES S-box: GF inversion -> affine transformation."""
    return affine_fwd(gf_inv(a))


# Build the full 256-entry forward S-box table
SBOX = [sbox(i) for i in range(256)]

# Sanity checks against known FIPS 197 values
assert SBOX[0x00] == 0x63, "S-box(0x00) should be 0x63"
assert SBOX[0x53] == 0xED, "S-box(0x53) should be 0xED"
assert SBOX[0x7C] == 0x10, "S-box(0x7C) should be 0x10"

print("Forward S-box built and verified against FIPS 197.")
print(f"  SBOX[0x00] = {SBOX[0x00]:#04x}  (expected 0x63)")
print(f"  SBOX[0x53] = {SBOX[0x53]:#04x}  (expected 0xed)")
print(f"  SBOX[0x7C] = {SBOX[0x7C]:#04x}  (expected 0x10)")

In [ ]:
# ============================================================
# Section 2 — Building the inverse S-box by table inversion
# ============================================================
# The simplest possible construction: because SBOX is a bijection
# (every input maps to a unique output), we can build INVSBOX by
# simply reversing every mapping.
#
# If SBOX[x] = y, then INVSBOX[y] = x.
#
# No new mathematics is required — just reading the forward table
# backward.

INVSBOX = [0] * 256
for x in range(256):
    y = SBOX[x]
    INVSBOX[y] = x

# Spot-checks: the inverse should undo the forward mapping
checks = [(0x00, 0x63), (0x53, 0xED), (0x7C, 0x10), (0x01, 0x7C), (0x02, 0x77)]

print("Inverse S-box construction — spot-checks")
print("-" * 50)
print(f"{'Forward':^25}  {'Inverse':^25}")
print(f"{'SBOX[x] = y':^25}  {'INVSBOX[y] = x':^25}")
print("-" * 50)
for x, y in checks:
    fwd_ok = SBOX[x] == y
    inv_ok = INVSBOX[y] == x
    print(f"SBOX[{x:#04x}] = {y:#04x}  {'✓' if fwd_ok else '✗'}  "
          f"  INVSBOX[{y:#04x}] = {INVSBOX[y]:#04x}  {'✓' if inv_ok else '✗'}")

print()
print("Key insight: INVSBOX is not a second unrelated table.")
print("It is the forward table read backward — each (x, y) pair")
print("in SBOX becomes a (y, x) pair in INVSBOX.")

In [ ]:
# ============================================================
# Section 3 — The inverse affine transformation
# ============================================================
# The forward affine is: s = A·b ⊕ c  (c = bits of 0x63)
# Its inverse is:        b = A⁻¹·s ⊕ A⁻¹·c
#
# The inverse affine formula, derived from the inverse matrix A⁻¹:
#
#   b_i = s_i ⊕ s_{(i+2)%8} ⊕ s_{(i+5)%8} ⊕ s_{(i+7)%8} ⊕ d_i
#
# where d = A⁻¹·c has bits (d₀…d₇) = (1,0,1,0,0,0,0,0) = 0x05.
#
# We implement it two ways and verify they agree:
#   Method A — via the bit formula above.
#   Method B — derived from the tables: inv_affine(s) = gf_inv(INVSBOX[s])
#              (because INVSBOX[s] = gf_inv(inv_affine(s)), so applying
#               gf_inv to both sides gives inv_affine(s) = gf_inv(INVSBOX[s])).

# Inverse affine constant: bits of 0x05, index 0 = LSB
CONST_INV = [1, 0, 1, 0, 0, 0, 0, 0]


def affine_inv_formula(s_byte: int) -> int:
    """Inverse affine via the explicit bit formula."""
    s = [(s_byte >> i) & 1 for i in range(8)]
    b = [s[i] ^ s[(i+2)%8] ^ s[(i+5)%8] ^ s[(i+7)%8] ^ CONST_INV[i]
         for i in range(8)]
    return sum(b[i] << i for i in range(8))


def affine_inv_table(s_byte: int) -> int:
    """Inverse affine derived from INVSBOX: inv_affine(s) = gf_inv(INVSBOX[s])."""
    return gf_inv(INVSBOX[s_byte])


# Verify both methods agree for all 256 values
mismatches = [
    s for s in range(256)
    if affine_inv_formula(s) != affine_inv_table(s)
]
print(f"Comparing formula vs table for all 256 values: ", end="")
if not mismatches:
    print("all match ✓")
else:
    print(f"MISMATCHES at: {[hex(s) for s in mismatches]}")

# Demonstrate the key example: inverse affine of 0xED → 0xCA
print()
print("Key example: inverting 0xED (the S-box output for 0x53)")
print("-" * 55)
s_byte = 0xED
s_bits = [(s_byte >> i) & 1 for i in range(8)]
print(f"Input s_byte = {s_byte:#04x} = {s_byte:08b}b")
print(f"Bits (s0..s7, LSB first): {s_bits}")
print(f"Inverse constant CONST_INV (d0..d7): {CONST_INV}  (0x05)")
print()
for i in range(8):
    bi = s_bits[i] ^ s_bits[(i+2)%8] ^ s_bits[(i+5)%8] ^ s_bits[(i+7)%8] ^ CONST_INV[i]
    print(f"  b{i} = s{i}({s_bits[i]}) ^ s{(i+2)%8}({s_bits[(i+2)%8]}) "
          f"^ s{(i+5)%8}({s_bits[(i+5)%8]}) ^ s{(i+7)%8}({s_bits[(i+7)%8]}) "
          f"^ d{i}({CONST_INV[i]}) = {bi}")
result = affine_inv_formula(s_byte)
print(f"\nResult: inv_affine({s_byte:#04x}) = {result:#04x}")
assert result == 0xCA, f"Expected 0xCA, got {result:#04x}"
print("Expected: 0xca (the intermediate value from forward path)  ✓")

In [ ]:
# ============================================================
# Section 4 — Step-by-step trace of the inverse S-box
# ============================================================
# The inverse S-box for input y:
#   Step 1: apply inverse affine  ->  intermediate
#   Step 2: apply GF(2^8) inverse ->  recovered original x
#
# We trace the canonical example: InvSBOX(0xED) should return 0x53.

def invsbox(y: int) -> int:
    """Inverse AES S-box: inverse affine -> GF(2^8) inversion."""
    return gf_inv(affine_inv_formula(y))


print("Step-by-step trace: InvSBOX(0xED)")
print("=" * 55)

y = 0xED
print(f"\nInput y = {y:#04x} (this is S-box(0x53) from Module 14)")

print("\n--- Step 1: Inverse affine transformation ---")
inter = affine_inv_formula(y)
print(f"  inv_affine({y:#04x}) = {inter:#04x}")
print(f"  This recovers the intermediate from the forward path.")
print(f"  In Module 14 we saw: 0x53 -> [field inv] -> {inter:#04x} -> [affine] -> {y:#04x}")
print(f"  The inverse affine returns us to that same intermediate: {inter:#04x} ✓")

print("\n--- Step 2: GF(2^8) multiplicative inversion ---")
recovered = gf_inv(inter)
verify = gf_mul(inter, recovered)
print(f"  gf_inv({inter:#04x}) = {recovered:#04x}")
print(f"  Verify: gf_mul({inter:#04x}, {recovered:#04x}) = {verify:#04x}  "
      f"{'✓' if verify == 1 else 'FAIL'}")
print(f"  Note: (a⁻¹)⁻¹ = a in GF(2^8), so gf_inv(0xCA) = 0x53 ✓")

print(f"\nResult: InvSBOX({y:#04x}) = {recovered:#04x}")
assert recovered == 0x53
print("Expected: 0x53  ✓")

print()
print("Compare forward and reverse paths side by side:")
print("-" * 55)
print(f"  Forward: 0x53 --[field inv]--> 0xCA --[affine]--> 0xED")
print(f"  Reverse: 0xED --[inv affine]--> 0xCA --[field inv]--> 0x53")
print()
print("The intermediate 0xCA appears in both paths.")
print("The reverse path is the forward path walked backward.")

In [ ]:
# ============================================================
# Section 5 — Proof: InvSBOX[SBOX[a]] == a for all 256 bytes
# ============================================================
# This is the computational proof of the round-trip identity:
#   S⁻¹(S(a)) = a   for every byte a in {0x00, ..., 0xFF}
#
# We check all 256 cases exhaustively.

print("Round-trip verification: InvSBOX[SBOX[a]] == a for all a")
print("=" * 55)

failures = []
for a in range(256):
    y = SBOX[a]          # forward S-box
    recovered = invsbox(y)  # inverse S-box
    if recovered != a:
        failures.append((a, y, recovered))

if not failures:
    print(f"\nAll 256 round-trips hold.  S⁻¹(S(a)) = a  ✓  for every byte.")
else:
    print(f"\nFAILURES ({len(failures)}):")
    for a, y, got in failures:
        print(f"  a={a:#04x}: S-box={y:#04x}, InvSBOX returned {got:#04x} (expected {a:#04x})")

# Also verify the table-based INVSBOX agrees with the computed invsbox()
invsbox_table_failures = [
    a for a in range(256) if INVSBOX[SBOX[a]] != a
]
print(f"\nTable-based INVSBOX round-trip failures: {len(invsbox_table_failures)}")
if not invsbox_table_failures:
    print("Table-based INVSBOX is also verified for all 256 bytes.  ✓")

print()
print("Sample round-trips (to make it concrete):")
print("-" * 55)
sample = [0x00, 0x53, 0x7C, 0x01, 0xFF, 0xAB, 0xF0, 0x80]
for a in sample:
    y = SBOX[a]
    r = INVSBOX[y]
    print(f"  {a:#04x} -> SBOX -> {y:#04x} -> INVSBOX -> {r:#04x}  "
          f"{'✓' if r == a else 'FAIL'}")

In [ ]:
# ============================================================
# Section 6 — Bijection verification
# ============================================================
# A function is reversible if and only if it is a bijection:
# every output value appears exactly once.
# We verify this for both SBOX and INVSBOX.

print("Bijection verification for SBOX and INVSBOX")
print("=" * 55)

# --- Forward S-box ---
sbox_outputs = sorted(set(SBOX))
sbox_is_bijection = len(sbox_outputs) == 256
print(f"\nForward SBOX:")
print(f"  Distinct output values: {len(sbox_outputs)} / 256")
print(f"  Is a bijection: {sbox_is_bijection}  {'✓' if sbox_is_bijection else '✗'}")

# --- Inverse S-box ---
invsbox_outputs = sorted(set(INVSBOX))
invsbox_is_bijection = len(invsbox_outputs) == 256
print(f"\nInverse INVSBOX:")
print(f"  Distinct output values: {len(invsbox_outputs)} / 256")
print(f"  Is a bijection: {invsbox_is_bijection}  {'✓' if invsbox_is_bijection else '✗'}")

# --- Check for fixed points in INVSBOX ---
invsbox_fixed = [a for a in range(256) if INVSBOX[a] == a]
print(f"\nFixed points of INVSBOX (where INVSBOX[a] == a): {len(invsbox_fixed)}")
print("(The forward S-box has no fixed points; the inverse S-box also has none.)")

# --- Visualise the output distribution ---
print()
print("Output frequency histogram (each value should appear exactly once):")
from collections import Counter
freq = Counter(INVSBOX)
max_freq = max(freq.values())
min_freq = min(freq.values())
print(f"  Min frequency: {min_freq}  Max frequency: {max_freq}")
if max_freq == 1 and min_freq == 1:
    print("  Every output appears exactly once — perfect bijection.  ✓")
else:
    dupes = [v for v, c in freq.items() if c > 1]
    print(f"  Duplicate outputs found: {dupes}")

In [ ]:
# ============================================================
# Section 7 — The shared intermediate value
# ============================================================
# One of the most important structural insights in this module:
# the value produced by the forward field-inverse step
# is EXACTLY the value recovered by the inverse-affine step.
#
# For any byte a:
#   forward path:  a -> gf_inv(a) -> affine_fwd(gf_inv(a)) = s
#   reverse path:  s -> affine_inv(s) -> gf_inv(affine_inv(s)) = a
#
# The intermediate  gf_inv(a)  =  affine_inv(s)
#
# This is what makes the reversal structurally clean.

print("Shared intermediate value: gf_inv(a) == affine_inv(S-box(a))")
print("=" * 60)

# Verify this identity holds for all 256 bytes
mismatches = []
for a in range(256):
    fwd_inter = gf_inv(a)            # intermediate in forward path
    s = SBOX[a]                      # S-box output
    rev_inter = affine_inv_formula(s) # intermediate in reverse path
    if fwd_inter != rev_inter:
        mismatches.append((a, fwd_inter, rev_inter))

if not mismatches:
    print("\ngf_inv(a) == affine_inv(SBOX[a])  for all 256 bytes  ✓")
else:
    print(f"Mismatches: {mismatches}")

# Print the comparison for a concrete sample
print()
print(f"{'Byte a':>8} | {'Forward inter':>14} | {'S-box out':>10} | {'Reverse inter':>14} | {'Match':>6}")
print("-" * 62)
sample = [0x00, 0x53, 0x7C, 0x01, 0xFF, 0x80, 0xAB, 0x3F]
for a in sample:
    fwd_i = gf_inv(a)
    s     = SBOX[a]
    rev_i = affine_inv_formula(s)
    match = '✓' if fwd_i == rev_i else '✗'
    print(f"  {a:#06x}  | {fwd_i:#14x} | {s:#10x} | {rev_i:#14x} | {match:>6}")

print()
print("The forward and reverse paths share one meeting point.")
print("That shared value is gf_inv(a) — the field inverse of the original byte.")

In [ ]:
# ============================================================
# Section 8 — Interactive explorer: round_trip_explore()
# ============================================================

def round_trip_explore(hex_val):
    """Display the full forward S-box and inverse S-box paths for any byte.

    Shows:
    - Forward: a -> [field inv] -> intermediate -> [affine] -> S-box output
    - Reverse: S-box output -> [inv affine] -> intermediate -> [field inv] -> a
    - Confirmation that recovered == original

    Parameters
    ----------
    hex_val : int or str
        Input byte, e.g. 0x53, '0x53', or 83.

    Examples
    --------
    >>> round_trip_explore(0x53)
    >>> round_trip_explore('0x00')
    >>> round_trip_explore(255)
    """
    if isinstance(hex_val, str):
        a = int(hex_val, 16) if hex_val.lower().startswith('0x') else int(hex_val, 16)
    else:
        a = int(hex_val)

    if not 0 <= a <= 255:
        print(f"Error: value must be 0x00–0xFF, got {a}")
        return

    inter   = gf_inv(a)          # forward field inverse
    s       = SBOX[a]            # S-box output
    inter2  = affine_inv_formula(s)  # reverse: after inv affine
    recov   = gf_inv(inter2)     # reverse: after second field inverse

    print(f"Round-trip exploration for byte {a:#04x} ({a:08b}b, decimal {a})")
    print("=" * 60)

    print(f"\n{'FORWARD S-BOX':^60}")
    print("-" * 60)
    if a == 0:
        print(f"  Stage 1 (field inverse):  {a:#04x} --> {inter:#04x}  "
              f"[special case: 0 maps to 0]")
    else:
        verify_fwd = gf_mul(a, inter)
        print(f"  Stage 1 (field inverse):  {a:#04x} --> {inter:#04x}  "
              f"[gf_mul({a:#04x}, {inter:#04x}) = {verify_fwd:#04x}  "
              f"{'✓' if verify_fwd == 1 else 'FAIL'}]")
    print(f"  Stage 2 (affine fwd):      {inter:#04x} --> {s:#04x}")
    print(f"  S-box({a:#04x}) = {s:#04x}")

    print(f"\n{'INVERSE S-BOX':^60}")
    print("-" * 60)
    print(f"  Stage 1 (inv affine):      {s:#04x} --> {inter2:#04x}  "
          f"[same intermediate as forward: {inter2 == inter}  "
          f"{'✓' if inter2 == inter else '✗'}]")
    if inter2 == 0:
        print(f"  Stage 2 (field inverse):  {inter2:#04x} --> {recov:#04x}  "
              f"[special case: 0 maps to 0]")
    else:
        verify_rev = gf_mul(inter2, recov)
        print(f"  Stage 2 (field inverse):  {inter2:#04x} --> {recov:#04x}  "
              f"[gf_mul({inter2:#04x}, {recov:#04x}) = {verify_rev:#04x}  "
              f"{'✓' if verify_rev == 1 else 'FAIL'}]")
    print(f"  InvSBOX({s:#04x}) = {recov:#04x}")

    match = recov == a
    print(f"\n{'ROUND-TRIP RESULT':^60}")
    print("-" * 60)
    print(f"  Original:  {a:#04x}")
    print(f"  Recovered: {recov:#04x}")
    print(f"  S⁻¹(S({a:#04x})) = {recov:#04x}  {'✓ Match' if match else '✗ MISMATCH'}")


# Demonstrate with four examples
for val in [0x53, 0x00, 0x7C, 0xFF]:
    round_trip_explore(val)
    print()

In [ ]:
# ============================================================
# Section 9 — Collision thought experiment
# ============================================================
# What happens to decryption if an S-box has duplicate outputs?
# We construct a deliberately broken S-box and demonstrate
# exactly why it cannot be inverted.

print("Collision thought experiment: a broken S-box")
print("=" * 55)

# Start with the real S-box and introduce one collision
BROKEN_SBOX = list(SBOX)
# Make two different inputs map to the same output
colliding_input_a = 0x10
colliding_input_b = 0x20
BROKEN_SBOX[colliding_input_b] = BROKEN_SBOX[colliding_input_a]   # collision!

collision_output = BROKEN_SBOX[colliding_input_a]
print(f"\nInjected collision:")
print(f"  BrokenSBOX[{colliding_input_a:#04x}] = {collision_output:#04x}")
print(f"  BrokenSBOX[{colliding_input_b:#04x}] = {collision_output:#04x}  <-- same output!")

# Check: is the broken S-box still a bijection?
distinct_outputs = len(set(BROKEN_SBOX))
print(f"\nDistinct outputs in broken S-box: {distinct_outputs} / 256")
print(f"(One output value appears twice; one output value never appears.)")

# Demonstrate the decryption failure
print(f"\nDecryption failure:")
print(f"  A ciphertext byte of {collision_output:#04x} could have come from EITHER:")
print(f"    - plaintext {colliding_input_a:#04x} (original mapping), OR")
print(f"    - plaintext {colliding_input_b:#04x} (collision mapping)")
print(f"  The inverse S-box cannot determine which. Information is lost.")
print(f"  Also: the byte that USED to map to where {colliding_input_b:#04x} now maps")
print(f"  has no output in the broken table — decryption of that output is impossible.")

print()
print("Conclusion: a reversible S-box cannot have duplicate outputs.")
print("AES avoids this entirely — the algebraic construction guarantees")
print("a bijection for all 256 byte values.")

## Section 10 — Summary and Bridge to Module 16

### What we proved

| Property | Verified |
|---|---|
| `InvSBOX[SBOX[a]] == a` for all 256 bytes | ✓ exhaustive check |
| INVSBOX is a bijection (256 distinct outputs) | ✓ |
| INVSBOX has no fixed points | ✓ |
| Inverse affine formula matches table derivation for all 256 inputs | ✓ |
| Forward and reverse paths share the same intermediate value | ✓ |

### The structure that makes reversal work

```
FORWARD:  a  --[gf_inv]--> inter --[affine_fwd]--> s
REVERSE:  s  --[affine_inv]--> inter --[gf_inv]--> a
                               ↑
                         same intermediate
```

Both paths pass through the same intermediate value `gf_inv(a)`. The inverse path is exactly the forward path walked backward — no new mathematics, just direction reversed.

### Why this matters for AES

- **Decryption uses InvSubBytes** (the inverse S-box) in place of SubBytes.
- Reversibility is not in tension with security: the S-box is non-linear and hard to invert *without the table*, but the table itself is fully public and algebraically determined.
- A block cipher that could only encrypt would be useless for almost all real applications.

### Bridge to Module 16 — Linear Transformations and Matrices over GF(2)

The affine transformation at the heart of the S-box is built from a matrix multiplication over GF(2). Module 16 develops the full language of **linear transformations and matrices over GF(2)** — the mathematics that makes the affine step (and its inverse) precise and computable. The same framework will reappear in **MixColumns**, where a 4×4 matrix over GF(2⁸) provides the diffusion layer in each AES round.

---

*Mathematics of Cryptography · Module 15 · Inverse S-box and Reversibility*